# ⚖️ Libratio Fleet - Multi-Agent GRPO Training
This notebook trains a tiny `Llama-3.1-8B` model using **GRPO (Reinforcement Learning)** to act as the Fleet Oversight Agent. It uses a lightning-fast standalone verifier to grade the LLM's JSON actions zero-shot, removing the need for a web server.

**Theme:** Multi-Agent Interactions + Fleet AI

In [ ]:
# 1. Install required packages (Unsloth, TRL) for fast RL training
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes datasets

In [ ]:
# 2. Load the Model using Unsloth (Super fast!)
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit", # Use 4-bit to fit in free Colab GPU
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# Add LoRA adapters so it can learn our specific hackathon tasks
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
# 3. Define the Standalone Verifier (Reward Function)
# Instead of calling a slow HTTP server, we verify the action statelessly!
import json

def fleet_oversight_reward(prompts, completions, **kwargs):
    rewards = []
    for prompt, completion in zip(prompts, completions):
        prompt_text = prompt[-1]["content"]
        action_text = completion[0]["content"]
        
        try:
            if "```json" in action_text:
                action_text = action_text.split("```json")[1].split("```")[0]
            action = json.loads(action_text)
            
            score = 0.01
            # Check if there was a 'null' in the prompt (indicating NaN crash)
            has_nan = "null" in prompt_text or "None" in prompt_text
            
            action_type = action.get("action_type", "")
            if has_nan:
                if action_type == "flag_instability":
                    score = 0.95  # Correctly flagged the crash
                else:
                    score = 0.10  # Missed the crash
            else:
                if action_type == "continue_monitoring":
                    score = 0.95  # Correctly continued monitoring
                else:
                    score = 0.10  # False alarm
                    
            rewards.append(score)
        except:
            rewards.append(0.01) # Penalize bad JSON format
            
    return rewards

In [ ]:
# 4. Generate Training Data (Mocking the Environment)
from datasets import Dataset
import random

stable_obs = {'task_id': 'fleet_oversight', 'model_trajectories': {'model_a': {'loss_window': [2.5, 2.4, 2.3]}, 'model_b': {'loss_window': [1.5, 1.4, 1.3]}}}
crash_obs = {'task_id': 'fleet_oversight', 'model_trajectories': {'model_a': {'loss_window': [2.5, 2.4, 2.3]}, 'model_b': {'loss_window': [1.5, None, None]}}}

training_data = []
for _ in range(100):
    obs = random.choice([stable_obs, crash_obs])
    training_data.append({
        "prompt": [
            {"role": "system", "content": "You are a Fleet AI Oversight Agent. Output valid JSON with 'action_type'."}, 
            {"role": "user", "content": f"Current environment state:\n{json.dumps(obs, default=str)}"}
        ]
    })

dataset = Dataset.from_list(training_data)
print(f"Generated {len(dataset)} training episodes!")

In [ ]:
# 5. Train the Agent using GRPO!
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir="fleet_outputs",
    learning_rate=5e-6,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    max_steps=50,
    logging_steps=5,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[fleet_oversight_reward],
    args=training_args,
    train_dataset=dataset,
)

print("Starting Fleet RL Training...")
trainer.train()

In [ ]:
# 6. Save the trained agent!
model.save_pretrained("libratio_fleet_lora")
print("Model saved! Hackathon phase complete 🚀")